import importlib
import spark_data_check
importlib.reload(spark_data_check)

from spark_data_check import SparkDataCheck
# Python file for data checking in Spark. 
#### Name: Cole Hammett
#### Date: 27th of March, 2026
#### Class: ST554: Analysis of Big Data
#### Purpose: This file contains functions for data checks on Spark DataFrames (null values, duplicates, and data types). 

# Part I: `SparkDataCheck` Example Usage

## Setup, Spark Session

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("SparkDataCheckDemo") \
    .getOrCreate()
spark.conf.set("spark.sql.ansi.enabled", "false")

print("Spark version:", spark.version)

## Import and Reload the Module

In [ ]:
import importlib
import spark_data_check
import pandas

importlib.reload(spark_data_check)
from spark_data_check import SparkDataCheck 

print("SparkDataCheck imported successfully.")

## 1. Load Air Quality Data via `from_csv`

In [ ]:
import pandas as pd

AIR_URL = "https://www4.stat.ncsu.edu/online/datasets/air.csv"

# Step 1: Download with pandas (handles HTTPS fine)
pdf = pd.read_csv(AIR_URL)

# Step 2: Create SparkDataCheck from the pandas DataFrame
sdc = SparkDataCheck.from_pandas(spark, pdf)

# Verify
print("Type:", type(sdc))
print("Schema:")
sdc.df.printSchema()
sdc.df.show(5)

### String Columns for Later 

In [ ]:
# Step 1: Extract Month from the Date column
sdc.df = sdc.df.withColumn(
    "Month",
    F.month(F.to_date(F.col("Date"), "m/d/yyyy"))
)

# Step 2: Now assign Season using the new Month column
sdc.df = sdc.df.withColumn(
    "Season",
    F.when(F.col("Month") == 5, "Spring")
     .when(F.col("Month").isin([6, 7, 8]), "Summer")
     .otherwise("Fall")
)

# Step 3: Add TempCat (note: your temp column is "T", not "Temp")
sdc.df = sdc.df.withColumn(
    "TempCat",
    F.when(F.col("T") >= 80, "Hot").otherwise("Cool")
)

print("Updated schema after adding engineered columns:")
sdc.df.printSchema()
sdc.df.show(8)

### Helper

In [ ]:
def fresh_sdc():
    """Return a clean SparkDataCheck built from the raw pandas DataFrame."""
    return SparkDataCheck.from_pandas(spark, pdf)

def fresh_sdc_with_strings():
    """Return a SparkDataCheck with Month, Season, and TempCat columns added."""
    obj = SparkDataCheck.from_pandas(spark, pdf)
    obj.df = obj.df.withColumn(
        "Month", F.month(F.to_date(F.col("Date"), "m/d/yyyy"))
    )
    obj.df = obj.df.withColumn(
        "Season",
        F.when(F.col("Month") == 5, "Spring")
         .when(F.col("Month").isin([6, 7, 8]), "Summer")
         .otherwise("Fall")
    )
    obj.df = obj.df.withColumn(
        "TempCat",
        F.when(F.col("T") >= 80, "Hot").otherwise("Cool")
    )
    return obj

print("Helper functions defined.")

## 2. `check_numeric_range`, 5 Examples


In [ ]:
# ---------- Example 1: Both lower AND upper bounds on CO(GT) ----------
# CO(GT) is carbon monoxide concentration in mg/m^3.
# Realistic hourly values should be roughly 0–15; anything outside that range
# is either a sensor error or the -200 sentinel.
sdc_nr1 = fresh_sdc()

sdc_nr1.check_numeric_range("CO(GT)", lower=0.0, upper=15.0)

print("CO(GT) in [0.0, 15.0], False rows are likely -200 sentinel values:")
sdc_nr1.df.select("CO(GT)", "CO(GT)_in_range").show(10)

In [ ]:
# ---------- Example 2: LOWER bound only on T (temperature, Celsius) ----------
# Temperature in this Italian dataset should never drop below -20°C.
# We only care about the lower bound, there is no meaningful upper limit to enforce.
sdc_nr2 = fresh_sdc()

sdc_nr2.check_numeric_range("T", lower=-20.0)   # no upper argument

print("T >= -20.0 °C (lower bound only):")
sdc_nr2.df.select("T", "T_in_range").show(5)

In [ ]:
# ---------- Example 3: UPPER bound only on RH (relative humidity, %) ----------
# Relative humidity is physically bounded at 100 %. We only supply upper here.
sdc_nr3 = fresh_sdc()

sdc_nr3.check_numeric_range("RH", upper=100.0)  # no lower argument

print("RH <= 100.0 % (upper bound only):")
sdc_nr3.df.select("RH", "RH_in_range").show(5)

In [ ]:
# ---------- Example 4: Message prints, passing a STRING column (Season) ----------
# Season is a string column. check_numeric_range should detect that, print a
# warning message, and return self WITHOUT appending any column.
sdc_nr4 = fresh_sdc_with_strings()

cols_before = sdc_nr4.df.columns
print("Attempting check_numeric_range on string column 'Season':")
sdc_nr4.check_numeric_range("Season", lower=0, upper=10)

# Confirm no new column was added
print("\nColumns before:", cols_before)
print("Columns after: ", sdc_nr4.df.columns)

In [ ]:
# ---------- Example 5: CHAINING two check_numeric_range calls ----------
# Methods return self, so calls can be chained. We check CO(GT) and T together,
# then call .df.show() once at the end to display both appended Boolean columns.
sdc_nr5 = fresh_sdc()

print("Chaining: CO(GT) in [0, 15] AND T in [-20, 45], both columns appended:")
sdc_nr5 \
    .check_numeric_range("CO(GT)", lower=0.0, upper=15.0) \
    .check_numeric_range("T", lower=-20.0, upper=45.0) \
    .df.select("CO(GT)", "CO(GT)_in_range", "T", "T_in_range") \
    .show(10)

## 3. `check_string_levels`, 5 Examples

In [ ]:
# ---------- Example 1: All valid Season levels supplied, all rows should be True ----------
sdc_sl1 = fresh_sdc_with_strings()

print("Season in {Spring, Summer, Fall}, every row should return True:")
sdc_sl1.check_string_levels("Season", ["Spring", "Summer", "Fall"]) \
       .df.select("Season", "Season_in_levels").show(8)

In [ ]:
# ---------- Example 2: Only a SUBSET of levels, Spring and Fall rows flag False ----------
# Restricting to just ["Summer"] means only observations from June, July, and August
# will be flagged True; all other months return False.
sdc_sl2 = fresh_sdc_with_strings()

print("Season in {Summer} only, Spring and Fall flagged as False:")
sdc_sl2.check_string_levels("Season", ["Summer"]) \
       .df.select("Month", "Season", "Season_in_levels").show(10)

In [ ]:
# ---------- Example 3: Message prints, passing a NUMERIC column (CO(GT)) ----------
# CO(GT) is a double column. check_string_levels should detect that and print a
# warning, leaving the DataFrame unchanged.
sdc_sl3 = fresh_sdc_with_strings()

cols_before = sdc_sl3.df.columns
print("Attempting check_string_levels on numeric column 'CO(GT)':")
sdc_sl3.check_string_levels("CO(GT)", ["1.0", "2.0", "3.0"])

print("\nColumns unchanged:", sdc_sl3.df.columns == cols_before)

In [ ]:
# ---------- Example 4: TempCat, only 'Cool' is in the levels set ----------
# Since T is in Celsius and never reaches 80°C in this dataset, TempCat is always
# "Cool". Checking ["Cool"] should therefore return True for every row.
sdc_sl4 = fresh_sdc_with_strings()

print("TempCat in {Cool}, all rows should be True (no 'Hot' readings exist):")
sdc_sl4.check_string_levels("TempCat", ["Cool"]) \
       .df.select("T", "TempCat", "TempCat_in_levels").show(5)

In [ ]:
# ---------- Example 5: CHAINING two check_string_levels calls ----------
# Check Season and TempCat in one chained expression, then call .df.show() once.
sdc_sl5 = fresh_sdc_with_strings()

print("Chaining: Season in {Summer} AND TempCat in {Cool, Hot}:")
sdc_sl5 \
    .check_string_levels("Season", ["Summer"]) \
    .check_string_levels("TempCat", ["Cool", "Hot"]) \
    .df.select("Month", "Season", "Season_in_levels", "TempCat", "TempCat_in_levels") \
    .show(8)

## 4. `check_missing`, 5 Examples

In [ ]:
# ---------- Example 1: Check CO(GT) for NULLs ----------
# The original dataset has some fully blank rows at the end of the Excel export;
# those parse as NULL when loaded through pandas → Spark.
sdc_cm1 = fresh_sdc()

print("check_missing on 'CO(GT)', True where value is NULL:")
sdc_cm1.check_missing("CO(GT)").df.select("CO(GT)", "CO(GT)_missing").show(10)

In [ ]:
# ---------- Example 2: Check Date for NULLs ----------
# Date is a string column, check_missing works on any dtype.
# The trailing blank rows in the export show up here as NULL dates.
sdc_cm2 = fresh_sdc()

print("check_missing on 'Date', string column, NULL where row is blank:")
sdc_cm2.check_missing("Date") \
       .df.select("Date", "Date_missing").show(10)

In [ ]:
# ---------- Example 3: Check T (temperature) for NULLs ----------
sdc_cm3 = fresh_sdc()

print("check_missing on 'T' (temperature in Celsius):")
sdc_cm3.check_missing("T") \
       .df.select("T", "T_missing").show(5)

In [ ]:
# ---------- Example 4: CHAINING check_missing on two columns ----------
# Chain CO(GT) and Date missing checks so both appended columns appear together.
sdc_cm4 = fresh_sdc()

print("Chaining check_missing on CO(GT) and Date:")
sdc_cm4 \
    .check_missing("CO(GT)") \
    .check_missing("Date") \
    .df.select("CO(GT)", "CO(GT)_missing", "Date", "Date_missing") \
    .show(10)

In [ ]:
# ---------- Example 5: Chain check_missing WITH check_numeric_range ----------
# Mixed-type chaining: first flag missing values in CO(GT), then flag out-of-range
# values in RH, all in one expression ending with .df.show().
sdc_cm5 = fresh_sdc()

print("Chain: check_missing('CO(GT)') → check_numeric_range('RH', lower=0, upper=100):")
sdc_cm5 \
    .check_missing("CO(GT)") \
    .check_numeric_range("RH", lower=0.0, upper=100.0) \
    .df.select("CO(GT)", "CO(GT)_missing", "RH", "RH_in_range") \
    .show(10)

## 5. `numeric_summary`, 5 Examples

In [ ]:
# ---------- Example 1: Single numeric column, no grouping (T, temperature) ----------
# numeric_summary returns a pandas DataFrame showing min and max of the column.
# Passing only col_name with no group_by gives a one-row global summary.
sdc_ns1 = fresh_sdc()

summary_t = sdc_ns1.numeric_summary(col_name="T")
print("Global min / max of T (°C):")
print(summary_t)

In [ ]:
# ---------- Example 2: Single numeric column, no grouping (CO(GT)) ----------
# CO(GT) contains -200 sentinel values for missing readings.
# The min should surface those sentinels, confirming the data needs cleaning.
sdc_ns2 = fresh_sdc()

summary_co = sdc_ns2.numeric_summary(col_name="CO(GT)")
print("Global min / max of CO(GT), negative min exposes -200 sentinels:")
print(summary_co)

In [ ]:
# ---------- Example 3: No column supplied, summarizes ALL numeric columns ----------
# When col=None the method automatically discovers every numeric column and reports
# its min and max. This is a quick data-quality snapshot of the entire dataset.
print("Min/max of ALL numeric columns (no grouping):")
sdc.numeric_summary()

In [ ]:
# ---------- Example 4: No column supplied, WITH grouping ----------
# Combines the two features above: summarizes every numeric column broken out by Season.
print("Min/max of ALL numeric columns grouped by Season:")
sdc.numeric_summary(group_by="Season")

In [ ]:
# ---------- Example 5: Message prints, passing a non-numeric column (Season) ----------
# Passing a string column should trigger the warning message and return None.
print("Passing non-numeric column 'Season' to numeric_summary:")
result = sdc.numeric_summary("Season")
print("Return value:", result)   # expected: None

## 6. `string_counts`, 5 Examples

`string_counts(col1, col2=None)` is also a summarization method it returns a pandas DataFrame with frequency counts for one or two string columns. With two columns it shows counts for every unique combination of levels. Passing a non-string column prints a warning.

In [ ]:
# ---------- Example 1: Single column, counts for each Season level ----------
# How many hourly observations fall in Spring, Summer, and Fall?
print("Counts per Season level:")
sdc.string_counts("Season")

In [ ]:
# ---------- Example 2: Single column, counts for TempCat ----------
# Since T is in Celsius and never reaches 80°C, we expect only the 'Cool' category.
print("Counts per TempCat level (all readings are below 80 °C):")
sdc.string_counts("TempCat")

In [ ]:
# ---------- Example 3: TWO columns, cross-tab of Season x TempCat ----------
# Shows every (Season, TempCat) combination and the number of rows in each.
# Since all TempCat values are 'Cool', this effectively becomes a Season-only count,
# which is a good sanity check.
print("Cross-tab counts for Season x TempCat combinations:")
sdc.string_counts("Season", "TempCat")

In [ ]:
# ---------- Example 4: Message prints, col1 is numeric (NOx(GT)) ----------
# NOx(GT) is a long integer column. string_counts should print a warning and
# return None (or an empty result) without crashing.
print("Attempting string_counts on numeric col1 'NOx(GT)':")
result = sdc.string_counts("NOx(GT)")
print("Return value:", result)

In [ ]:
# ---------- Example 5: Message prints, col2 is numeric (T) ----------
# col1 is valid ('Season'), but col2 is numeric ('T').
# The method should detect the invalid second column and print a message.
print("Attempting string_counts with valid col1='Season' and numeric col2='T':")
result = sdc.string_counts("Season", "T")
print("Return value:", result)

## 7. Loading via `from_pandas`, 1 Example

The `from_pandas` classmethod allows constructing a `SparkDataCheck` object from a pre-existing standard `pandas` DataFrame, useful when the data has already been read or pre-processed with pandas. Below we read the same air quality CSV with plain `pandas`, create a new `SparkDataCheck` instance, and run one method to confirm it behaves identically.

In [ ]:
# Read the CSV with standard pandas
pdf_fp = pd.read_csv(AIR_URL)

# Create a SparkDataCheck instance via the from_pandas classmethod
sdc_fp = SparkDataCheck.from_pandas(spark, pdf_fp)

print("Instance type:", type(sdc_fp))
print("Row count:", sdc_fp.df.count())

# Demonstrate one method call: check if RH (relative humidity) is in [0, 100]
print("\nRH in [0, 100] (from_pandas instance):")
sdc_fp.check_numeric_range("RH", lower=0.0, upper=100.0) \
      .df.select("RH", "RH_in_range").show(5)

---

# Part II: NFL Quarterback Analysis

- Task: Analyzing weekly NFL passing statistics for all quarterbacks who played regular-season games between the 2005 and 2023 seasons. The entire analysis was done twice, once using the pandas-on-Spark API and once using the Spark SQL DataFrame API
- Goal: is to identify the season-level leaders in completion percentage and touchdown-to-interception ratio, both measures of QB efficiency.

## 1. pandas-on-Spark Analysis

The pandas-on-Spark API to write nearly identical code to standard pandas while executing on the Spark cluster. There are no Spark SQL expressions are used in this subsection.

In [ ]:
import pyspark.pandas as ps

# Read the weekly NFL data from a local upload on JupyterHub.
# pyspark.pandas.read_csv() wraps Spark's CSV reader but returns a pandas-on-Spark DataFrame.
psdf = ps.read_csv("weekly_nfl_data.csv")

print("Type:", type(psdf))

In [ ]:
# Check out the first 5 rows to understand the structure of the dataset.
# Each row represents one player's stats for a single week and season.
print("First 5 rows:")
psdf.head(5)

The dataset is structured at the player-week-season level. Each row captures one player's in-game statistics for a single week, including passing, rushing, and receiving metrics. I focused exclusively on the passing columns for quarterbacks.

In [ ]:
# Report all column names so we can identify which columns to keep.
print("All columns:")
print(psdf.columns.tolist())

The dataset contains many columns covering rushing, receiving, and special teams in addition to passing. For this analysis we retain only the eight columns relevant to quarterback passing performance.

In [ ]:
# Filter rows: position == QB, regular season only, seasons 2005-2023 (inclusive).
# Select only the eight columns we need for the passing analysis.
stat_cols_ps = ["completions", "attempts", "passing_yards", "passing_tds", "interceptions"]
keep_cols_ps = ["player_display_name", "season", "week"] + stat_cols_ps

qb_ps = psdf[
    (psdf["position"] == "QB") &
    (psdf["season_type"] == "REG") &
    (psdf["season"] >= 2005) &
    (psdf["season"] <= 2023)
][keep_cols_ps]

print("Shape after filtering:", qb_ps.shape)
qb_ps.head(5)

After filtering we retain only QB regular-season observations between 2005 and 2023, a span of 19 NFL seasons. We now have one row per player per week.

In [ ]:
# Aggregate to one row per player-season by computing both the sum and the mean
# of each of the five passing statistics.
qb_agg_ps = qb_ps.groupby(["player_display_name", "season"]).agg(
    sum_completions   = ("completions",    "sum"),
    mean_completions  = ("completions",    "mean"),
    sum_attempts      = ("attempts",       "sum"),
    mean_attempts     = ("attempts",       "mean"),
    sum_passing_yards = ("passing_yards",  "sum"),
    mean_passing_yards= ("passing_yards",  "mean"),
    sum_passing_tds   = ("passing_tds",    "sum"),
    mean_passing_tds  = ("passing_tds",    "mean"),
    sum_interceptions = ("interceptions",  "sum"),
    mean_interceptions= ("interceptions",  "mean"),
).reset_index()

print("Rows after aggregation:", len(qb_agg_ps))
qb_agg_ps.head(5)

The aggregation collapses from week-level rows to one row per player-season. The `sum_*` columns give season totals while `mean_*` columns give per-game averages, both perspectives are useful for evaluating performance.

In [ ]:
# Derive the two efficiency metrics.
#
# completion_percentage = season completions / season attempts
# td_int_ratio          = season TDs / season interceptions
#
# NOTE (pandas-on-Spark behaviour): when sum_interceptions == 0, dividing by
# zero produces inf rather than NULL.  This differs from Spark SQL, see Part II-2
qb_agg_ps["completion_percentage"] = (
    qb_agg_ps["sum_completions"] / qb_agg_ps["sum_attempts"]
)
qb_agg_ps["td_int_ratio"] = (
    qb_agg_ps["sum_passing_tds"] / qb_agg_ps["sum_interceptions"]
)

print("New columns added. Sample:")
qb_agg_ps[["player_display_name", "season",
            "completion_percentage", "td_int_ratio"]].head(5)

`completion_percentage` is a well-defined ratio for any QB who attempted at least one pass. `td_int_ratio` is undefined when a QB threw zero interceptions, in pandas-on-Spark this produces `inf` (positive infinity), which will surface prominently when we sort below. This is an important difference from the Spark SQL behaviour documented in Part II-2.

In [ ]:
# Filter to player-season combinations with at least 50 attempts.
# This removes spot-starters and prevents very small sample sizes from
# dominating the leaderboards.
qb_filtered_ps = qb_agg_ps[qb_agg_ps["sum_attempts"] >= 50]

print("Rows with >= 50 attempts:", len(qb_filtered_ps))

In [ ]:
# Sort by completion_percentage descending and show the top 40 player-seasons.
print("Top 40 player-seasons by completion percentage (pandas-on-Spark):")
qb_filtered_ps.sort_values("completion_percentage", ascending=False).head(40)

Modern QBs dominate the completion percentage leaderboard, reflecting the league-wide shift toward short, high-percentage passing schemes. Elite pocket passers such as Drew Brees and later Patrick Mahomes and Justin Herbert appear repeatedly. Completion percentages above 70% were virtually unheard of before 2015 but are now common among top starters.

In [ ]:
# Sort by td_int_ratio descending and show the top 40 player-seasons.
# Note: rows where sum_interceptions == 0 will show inf and appear at the top.
print("Top 40 player-seasons by TD-to-INT ratio (pandas-on-Spark):")
qb_filtered_ps.sort_values("td_int_ratio", ascending=False).head(40)

pandas-on-Spark division-by-zero behaviour: When a QB threw zero interceptions during a season, `sum_interceptions == 0`, and the division produces `inf`. Those rows sort to the very top of the leaderboard. Whether to exclude or handle these cases depends on the analytical goal, they represent genuinely excellent ball security but make direct numerical comparison difficult.

Among finite ratios, top seasons tend to belong to QBs who were both prolific passers (many TDs) and extremely careful (few interceptions), characteristics often associated with late-career Drew Brees, prime Aaron Rodgers, and recent Patrick Mahomes seasons.

---

## 2: Spark SQL Analysis

We now repeat the exact same analysis using the Spark SQL DataFrame API, `pyspark.sql` style, which uses method chains of `.filter()`, `.select()`, `.groupBy().agg()`, and `.withColumn()` rather than the pandas-like syntax above. The results should be relatively identical with the exception when computing `td_int_ratio`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
spark.conf.set("spark.sql.ansi.enabled", "false")

# Re-use the existing SparkSession; reading the NFL CSV with Spark's native reader.
sdf_nfl = spark.read.load(
    "weekly_nfl_data.csv",
    format="csv",
    sep=",",
    inferSchema="true",
    header="true"
)

print("Schema:")
sdf_nfl.printSchema()

In [ ]:
# Check out the first 5 rows.
print("First 5 rows (Spark SQL):")
sdf_nfl.show(5)

The same player-week-season structure is visible. The column names match the pandas-on-Spark section, so the same filtering and selection logic applies.

In [ ]:
# Report all column names.
print("All columns (Spark SQL):")
print(sdf_nfl.columns)

In [ ]:
# Filter to QB regular-season rows in seasons 2005-2023 and select the
# eight columns needed for passing analysis.
stat_cols_sql = ["completions", "attempts", "passing_yards", "passing_tds", "interceptions"]

qb_sdf = (
    sdf_nfl
    .filter(
        (F.col("position")    == "QB")  &
        (F.col("season_type") == "REG") &
        (F.col("season")      >= 2005)  &
        (F.col("season")      <= 2023)
    )
    .select("player_display_name", "season", "week", *stat_cols_sql)
)

print("Row count after filtering:", qb_sdf.count())
qb_sdf.show(5)

The Spark SQL `.filter()` method uses `F.col()` expressions rather than the bracket-indexing syntax of pandas-on-Spark, but the logical conditions are identical.

In [ ]:
# Aggregate to player-season level: compute sum and mean of each passing stat.
qb_agg_sdf = (
    qb_sdf
    .groupBy("player_display_name", "season")
    .agg(
        F.sum("completions").alias("sum_completions"),
        F.mean("completions").alias("mean_completions"),
        F.sum("attempts").alias("sum_attempts"),
        F.mean("attempts").alias("mean_attempts"),
        F.sum("passing_yards").alias("sum_passing_yards"),
        F.mean("passing_yards").alias("mean_passing_yards"),
        F.sum("passing_tds").alias("sum_passing_tds"),
        F.mean("passing_tds").alias("mean_passing_tds"),
        F.sum("interceptions").alias("sum_interceptions"),
        F.mean("interceptions").alias("mean_interceptions"),
    )
)

print("Row count after aggregation:", qb_agg_sdf.count())
qb_agg_sdf.show(5)

Spark SQL's `.groupBy().agg()` produces the same player-season table as the pandas-on-Spark `groupby().agg()` call, but requires explicit `.alias()` calls rather than named keyword arguments.

In [ ]:
# Derive the two efficiency metrics using .withColumn().
#
# completion_percentage: straightforward division, well-defined for all QBs.
#
# td_int_ratio: IMPORTANT DIFFERENCE FROM PANDAS-ON-SPARK
#   In Spark SQL (ANSI mode off, the JupyterHub default), dividing an integer by
#   zero returns NULL rather than inf.  To make this explicit and avoid silent
#   surprises we use F.when() to produce NULL explicitly when sum_interceptions == 0.
qb_agg_sdf = (
    qb_agg_sdf
    .withColumn(
        "completion_percentage",
        F.col("sum_completions") / F.col("sum_attempts")
    )
    .withColumn(
        "td_int_ratio",
        F.when(
            F.col("sum_interceptions") == 0,
            F.lit(None)
        ).otherwise(
            F.col("sum_passing_tds") / F.col("sum_interceptions")
        )
    )
)

print("New columns added. Sample:")
qb_agg_sdf.select(
    "player_display_name", "season",
    "completion_percentage", "td_int_ratio"
).show(5)

### Division by Zero Issue

In the pandas-on-Spark section, QBs who threw zero interceptions received `td_int_ratio = inf`, causing them to sort to the top of the leaderboard. In the Spark SQL section we explicitly return `null` via `F.when(...).otherwise(...)`. This means those rows will sort to the bottom (or be excluded) when ordering in most Spark SQL contexts. The `F.when()` pattern is best practice for production Spark pipelines because it makes the handling of edge cases transparent and portable.

In [ ]:
# Filter to player-seasons with at least 50 attempts.
qb_filtered_sdf = qb_agg_sdf.filter(F.col("sum_attempts") >= 50)

print("Row count with >= 50 attempts:", qb_filtered_sdf.count())

In [ ]:
# Sort by completion_percentage descending and show the top 40 player-seasons.
print("Top 40 player-seasons by completion percentage (Spark SQL):")
qb_filtered_sdf \
    .orderBy(F.col("completion_percentage").desc()) \
    .show(40)

The Spark SQL completion percentage leaderboard should match the pandas-on-Spark result, the same player-seasons should appear in the same order, confirming that both APIs produce consistent aggregation results.

In [ ]:
# Sort by td_int_ratio descending and show the top 40 player-seasons.
# Rows with td_int_ratio = NULL (zero interceptions) sort to the bottom
# by default in Spark SQL, so the top of this list shows finite ratios only.
print("Top 40 player-seasons by TD-to-INT ratio (Spark SQL):")
qb_filtered_sdf \
    .orderBy(F.col("td_int_ratio").desc_nulls_last()) \
    .show(40)

Because `null` values sort to the bottom (we use `.desc_nulls_last()` to make this explicit), the Spark SQL leaderboard shows only QBs with at least one interception, making the finite ratios directly comparable. Contrast this with the pandas-on-Spark leaderboard where `inf` values floated to the top and required the reader to identify them manually.

Season-level leaders in TD:INT ratio tend to be QBs who combined high touchdown volume with exceptional ball security, qualities that define the best seasons from Aaron Rodgers, Tom Brady, and Patrick Mahomes. A ratio above 4:1 for a full season with 50+ attempts is considered elite by modern standards.